# Time Series Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense
from ESN import ESN
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
data_1950 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1950_c20170120.csv')
print("\n1950s data shape:", data_1950.shape)
print(data_1950.head())
data_1960 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1960_c20160223.csv')
print("\n1960s data shape:", data_1960.shape)
print(data_1960.head())
data_1970 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1970_c20160223.csv')
print("\n1970s data shape:", data_1970.shape)
print(data_1970.head())
data_1980 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1980_c20170717.csv')
print("\n1980s data shape:", data_1980.shape)
print(data_1980.head())
data_1990 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1990_c20170717.csv')
print("\n1990s data shape:", data_1990.shape)
print(data_1990.head())
data_2000 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d2000_c20200707.csv')
print("\n2000s data shape:", data_2000.shape)
print(data_2000.head())
data_2010 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d2010_c20200922.csv')
print("\n2010s data shape:", data_2010.shape)
print(data_2010.head())
data_2020 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d2020_c20201216.csv')
print("\n2020s data shape:", data_2020.shape)
print(data_2020.head())

In [ ]:
data = pd.concat([data_1950, data_1960, data_1970, data_1980, data_1990, data_2000, data_2010, data_2020])
print("\nCombined data shape:", data.shape)
print("\nMissing values")
print(data.isnull().sum())

In [ ]:
features = ["BEGIN_DATE_TIME", "MAGNITUDE", "EVENT_TYPE", "BEGIN_LAT", "BEGIN_LON"]
data = data[features]
print("\nMissing values after")
print(data.isnull().sum())
plt.figure(figsize=(10, 6))
sns.heatmap(data.isnull(), yticklabels=False, cbar=True, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
data.fillna(method='ffill', inplace=True)
print("\nMissing values after filling")
print(data.isnull().sum())

In [ ]:
data['BEGIN_DATE_TIME'] = pd.to_datetime(data['BEGIN_DATE_TIME'], errors='coerce')
print("\nInvalid dates (NaT values):", data['BEGIN_DATE_TIME'].isnull().sum())
data = data.dropna(subset=['BEGIN_DATE_TIME'])
print("\nData shape after removing invalid dates", data.shape)
data['YEAR'] = data['BEGIN_DATE_TIME'].dt.year
data['MONTH'] = data['BEGIN_DATE_TIME'].dt.month
data['DAY'] = data['BEGIN_DATE_TIME'].dt.day

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(131)
data['YEAR'].value_counts().sort_index().plot(kind='line')
plt.title('Events by Year')
plt.subplot(132)
data['MONTH'].value_counts().sort_index().plot(kind='bar')
plt.title('Events by Month')
plt.subplot(133)
data['DAY'].value_counts().sort_index().plot(kind='line')
plt.title('Events by Day')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(data['BEGIN_LON'], data['BEGIN_LAT'], alpha=0.1)
plt.title('Geographical Distribution')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
data[['MAGNITUDE', 'BEGIN_LAT', 'BEGIN_LON']] = scaler.fit_transform(data[['MAGNITUDE', 'BEGIN_LAT', 'BEGIN_LON']])
plt.figure(figsize=(15, 5))
plt.subplot(131)
sns.histplot(data['MAGNITUDE'], kde=True)
plt.title('Scaled Magnitude Distribution')
plt.subplot(132)
sns.histplot(data['BEGIN_LAT'], kde=True)
plt.title('Scaled Latitude Distribution')
plt.subplot(133)
sns.histplot(data['BEGIN_LON'], kde=True)
plt.title('Scaled Longitude Distribution')
plt.tight_layout()
plt.show()

In [ ]:
X = data[['MAGNITUDE', 'BEGIN_LAT', 'BEGIN_LON', 'YEAR', 'MONTH', 'DAY']].values
y = data['MAGNITUDE'].values
print("\nFinal X shape:", X.shape)
print("Final y shape:", y.shape)

In [ ]:
def load_and_preprocess_data():
    data_1950 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1950_c20170120.csv')
    data_1960 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1960_c20160223.csv')
    data_1970 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1970_c20160223.csv')
    data_1980 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1980_c20170717.csv')
    data_1990 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d1990_c20170717.csv')
    data_2000 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d2000_c20200707.csv')
    data_2010 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d2010_c20200922.csv')
    data_2020 = pd.read_csv(r'StormEvents_details-ftp_v1.0_d2020_c20201216.csv')

    data = pd.concat([data_1950, data_1960, data_1970, data_1980, data_1990, data_2000, data_2010, data_2020])

    features = ["BEGIN_DATE_TIME", "MAGNITUDE", "EVENT_TYPE", "BEGIN_LAT", "BEGIN_LON"]
    data = data[features]
    data.fillna(method='ffill', inplace=True)
    data['BEGIN_DATE_TIME'] = pd.to_datetime(data['BEGIN_DATE_TIME'], errors='coerce')
    data = data.dropna(subset=['BEGIN_DATE_TIME'])
    data['YEAR'] = data['BEGIN_DATE_TIME'].dt.year
    data['MONTH'] = data['BEGIN_DATE_TIME'].dt.month
    data['DAY'] = data['BEGIN_DATE_TIME'].dt.day

    scaler = MinMaxScaler(feature_range=(0, 1))
    data[['MAGNITUDE', 'BEGIN_LAT', 'BEGIN_LON']] = scaler.fit_transform(data[['MAGNITUDE', 'BEGIN_LAT', 'BEGIN_LON']])

    X = data[['MAGNITUDE', 'BEGIN_LAT', 'BEGIN_LON', 'YEAR', 'MONTH', 'DAY']].values
    y = data['MAGNITUDE'].values

    return X, y, scaler

In [ ]:
def train_test_split(X, y, test_size=0.2, validation_size=0.2):
    total_size = len(X)
    test_size = int(total_size * test_size)
    validation_size = int(total_size * validation_size)

    X_train = X[:-test_size-validation_size]
    y_train = y[:-test_size-validation_size]
    X_val = X[-test_size-validation_size:-test_size]
    y_val = y[-test_size-validation_size:-test_size]
    X_test = X[-test_size:]
    y_test = y[-test_size:]

    return X_train, y_train, X_val, y_val, X_test, y_test


In [ ]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R2: {r2:.4f}")

    plt.plot(y_test, label='Actual')
    plt.plot(predictions, label='Predicted')
    plt.legend()
    plt.title("Actual vs Predicted")
    plt.show()


In [ ]:
def create_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(50, activation='relu', input_shape=input_shape))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model


In [ ]:
def create_bi_lstm_model(input_shape):
    model = Sequential()
    model.add(Bidirectional(LSTM(50, activation='relu'), input_shape=input_shape))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model


In [ ]:
X, y, scaler = load_and_preprocess_data()
X_train, y_train, X_val, y_val, X_test, y_test = train_test_split(X, y)

X_train_reshaped = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_val_reshaped = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))
X_test_reshaped = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

lstm_model = create_lstm_model((X_train_reshaped.shape[1], X_train_reshaped.shape[2]))
lstm_model.fit(X_train_reshaped, y_train, epochs=30, batch_size=32, validation_data=(X_val_reshaped, y_val))
print("LSTM Model Evaluation:")

evaluate_model(lstm_model, X_test_reshaped, y_test)

bi_lstm_model = create_bi_lstm_model((X_train_reshaped.shape[1], X_train_reshaped.shape[2]))
bi_lstm_model.fit(X_train_reshaped, y_train, epochs=30, batch_size=32, validation_data=(X_val_reshaped, y_val))
print("Bi-LSTM Model Evaluation:")
evaluate_model(bi_lstm_model, X_test_reshaped, y_test)

esn_model = ESN(n_inputs=X_train.shape[1], n_outputs=1, n_reservoir=200)
esn_model.fit(X_train, y_train)
esn_predictions = esn_model.predict(X_test)
print("ESN Model Evaluation:")
rmse = np.sqrt(mean_squared_error(y_test, esn_predictions))
mae = mean_absolute_error(y_test, esn_predictions)
r2 = r2_score(y_test, esn_predictions)
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2: {r2:.4f}")

plt.plot(y_test, label='Actual')
plt.plot(esn_predictions, label='ESN Predicted')
plt.legend()
plt.title("Actual vs ESN Predicted")
plt.show()
